In [11]:
import numpy as np
import pandas as pd
import os
import glob


In [2]:
parquet_files = glob.glob("../data/processed/*.parquet")
merged = pd.read_parquet("../data/processed/loans_master.parquet")

merged.head()

,loan_id,issue_date,issue_year,issue_month,loan_amnt_inr,funded_amnt_inr,loan_term_months,int_rate_pct,installment_inr,annual_installment_inr,...,pymnt_plan,hardship_flag,initial_list_status,disbursement_method,verification_status,rbi_repo_rate_pct,gdp_growth_pct,cpi_inflation_pct,rate_spread_pct,real_interest_rate_pct
0,LN000000001,Feb-2016,2016,2,80678.0,74992.0,36,14.91,2793.17,33518.0,...,N,N,w,DIRECT_PAY,Source Verified,6.25,8.2,4.5,8.66,10.410000
1,LN000000002,May-2024,2024,5,274166.0,265041.0,36,7.00,8465.45,101585.0,...,N,N,w,CASH,Verified,6.50,6.8,4.9,0.50,2.100000
2,LN000000003,Dec-2021,2021,12,59603.0,54423.0,60,13.34,1366.55,16399.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,8.7,5.1,9.34,8.240000
3,LN000000004,Nov-2020,2020,11,246313.0,224181.0,84,24.07,6088.88,73067.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,-6.6,6.2,20.07,17.870001
4,LN000000005,Jul-2013,2013,7,101471.0,95361.0,60,8.52,2082.81,24994.0,...,Y,N,w,CASH,Verified,7.75,6.4,10.9,0.77,-2.380000


In [3]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 27 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   loan_id                 object  
 1   issue_date              category
 2   issue_year              int16   
 3   issue_month             int8    
 4   loan_amnt_inr           float32 
 5   funded_amnt_inr         float32 
 6   loan_term_months        int8    
 7   int_rate_pct            float32 
 8   installment_inr         float64 
 9   annual_installment_inr  float32 
 10  grade                   category
 11  sub_grade               category
 12  loan_purpose            category
 13  state_code              category
 14  region                  category
 15  urban_index             float32 
 16  application_type        category
 17  pymnt_plan              category
 18  hardship_flag           category
 19  initial_list_status     category
 20  disbursement_method     category
 21  verifica

In [ ]:
join_summary = []
for f in parquet_files:
    key=os.path.basename(f).replace(".parquet","")
    if key == "loans_master":
        continue
    df=pd.read_parquet(f)

    # For every loan_id in merged, does it exist in df?    
    orphan_count=(~merged["loan_id"].isin(df["loan_id"])).sum()
    
    merged = merged.merge(df,on="loan_id",how="left")

    join_summary.append({
        "table":key,
        "row_after_join":merged.shape[0],
        "orphans":orphan_count
    })
    print(f" + {key:35s} -> shape: {merged.shape}")
join_summary = pd.DataFrame(join_summary)
markdown_table = join_summary.to_markdown(index=False)
with open ("../report/figures/data_acqu_join_clean.md","a") as f:
    f.write("\n\n## Join Summary\n\n")
    f.write(markdown_table)


 + customer_bureau                     -> shape: (2000000, 56)
 + loan_performance                    -> shape: (2000000, 67)
 + payment_history                     -> shape: (2000000, 84)
 + branch_region_economy               -> shape: (2000000, 102)
 + monthly_emi_track                   -> shape: (2000000, 124)
 + loan_enquiry_bureau                 -> shape: (2000000, 147)
 + credit_card_behavior                -> shape: (2000000, 163)
 + collateral_assets                   -> shape: (2000000, 182)


### Saving the merged dataset

In [2]:
# merged.to_parquet("../data/merged/final_merged_dataset.parquet",index=False)
merged=pd.read_parquet("../data/merged/final_merged_dataset.parquet")

In [3]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Columns: 182 entries, loan_id to business_asset_val_inr
dtypes: category(42), float32(55), float64(29), int16(3), int32(1), int8(49), object(3)
memory usage: 1.1+ GB


In [4]:
merged.duplicated().sum()

np.int64(0)

In [5]:
missing_summary=merged.isna().sum()
missing_summary[missing_summary>0].sort_values(ascending=False)

vehicle_type                 1879789
property_city_tier           1679672
property_type                1679672
mths_since_last_record       1598332
ltv_ratio_pct                1472196
valuation_agency             1370048
collateral_type              1258896
charge_type                  1258896
mths_since_last_delinq       1098640
primary_cc_bank               560528
primary_card_type             560528
top_spend_category            560528
income_doc_type               308704
collateral_score              298379
mort_acc                      260330
il_util_pct                   200167
emp_length_years              180539
bc_util_pct                   160397
cash_advance_inr              160223
cc_payment_score              160149
branch_sanction_rate          140557
revol_util_pct                140097
loan_officer_exp_years        139366
avg_payment_delay_days        120364
emi_bank_name                 120236
collection_recovery_fee       120086
pdc_count                     120057
p

In [ ]:
# merged.describe(include="all").T.to_excel("../report/summaries/descriptive_stats.xlsx")

In [ ]:
# merged.describe().T.to_excel("../report/summaries/numerical_col_stats.xlsx")

### Dirty Flag

In [36]:
# Much percentage fields should satisfy 0 <= value <= 100
pct_cols=[c for c in merged.columns if "pct" in c]
# print(pct_cols)

for col in pct_cols:
    cnt=(
        (merged[col]<0) | (merged[col]>100)
    ).sum()

    if cnt>0:
        print(col,cnt)


gdp_growth_pct 139881
rate_spread_pct 23286
real_interest_rate_pct 76546
rejection_rate_pct 3168
ltv_ratio_pct 28700


Column
1. rejection_rate_pct-Negative rejection_rates make no sense
2. ltv (loan to value) ratio - negative almost certainly invalid

In [53]:
# Inr field should not be negative
# Chekcing INR fileds
inr_cols = [c for c in merged.columns if "inr" in c]

for col in inr_cols:
    cnt=((merged[col]<0)).sum()

    if cnt>0:
        print(col,cnt)

In [37]:
# Assignments frequently inject [-999,999,9999,99999]
# numbers like -999, 999, 9999, and 99999 are used as sentinel values or missing data placeholders.

num_cols = merged.select_dtypes(include="number").columns

col="col_name"
val="value"
cnt="count"

print(f"{col:25s}{val:15s}{cnt:15s}")

for col in num_cols:
    for val in [-999,999,9999,99999]:
        cnt=(merged[col]==val).sum()

        if cnt>0:
            print(f"{col:<25s}{val:<15d}{cnt:<15d}")


col_name                 value          count          
loan_amnt_inr            99999          11             
funded_amnt_inr          99999          6              
installment_inr          999            1              
annual_installment_inr   9999           1              
annual_installment_inr   99999          4              
revol_bal_inr            999            2              
revol_bal_inr            9999           2              
avg_cur_bal_inr          999            1              
provision_inr            999            9              
total_rec_prncp_inr      99999          11             
last_pymnt_amnt_inr      999            1              
last_pymnt_amnt_inr      9999           1              
expected_loss_inr        999            2              
installment_due_inr      999            1              
total_emi_due_inr        99999          3              
emi_overdue_inr          999            2              
emi_advance_paid_inr     999            1       

In [38]:
for col in [
    "property_area_sqft",
    "avg_monthly_cc_spend_inr",
    "cash_advance_inr"
]:
    print("\n", col)
    print(merged[col].value_counts().head(20))


 property_area_sqft
property_area_sqft
0       1679672
3456        105
4506        104
3396        103
4125        101
3985        100
2134         99
3535         99
764          98
4960         98
2499         98
514          97
4271         97
4888         96
4833         96
3075         96
1980         96
3511         96
2764         95
3972         95
Name: count, dtype: int64

 avg_monthly_cc_spend_inr
avg_monthly_cc_spend_inr
0.0       560528
2006.0       169
2471.0       160
2050.0       159
2478.0       157
2307.0       155
2360.0       153
2425.0       153
2753.0       152
1727.0       151
3517.0       151
1896.0       150
1656.0       150
2178.0       150
2218.0       149
1755.0       148
1893.0       148
1857.0       148
2368.0       148
1918.0       148
Name: count, dtype: int64

 cash_advance_inr
cash_advance_inr
0.0       1442038
537.0         118
551.0         115
664.0         113
944.0         112
708.0         111
919.0         111
1052.0        110
758.0         10

#### Column:
Most houses are 1200-2500 sqft
3. property_area_sqft - 999 need to inspect,property_area_sqft=0 for 1679672 alomst 84% value ?
4. avg_monthly _cc_spend_inr - 999, 86 records/9999-55 records,pattern ? avg_monthly_cc_spend_inr = 0 ?
5. cash_advance_inr - 999

In [39]:
obj_cols = merged.select_dtypes(include="object").columns

for col in obj_cols:
    print("\n", col)
    print(
        merged[col]
        .value_counts(dropna=False)
        .head(20)
    )


 loan_id
loan_id
LN000000001    1
LN000000002    1
LN000000003    1
LN000000004    1
LN000000005    1
LN000000006    1
LN000000007    1
LN000000008    1
LN000000009    1
LN000000010    1
LN000000011    1
LN000000012    1
LN000000013    1
LN000000014    1
LN000000015    1
LN000000016    1
LN000000017    1
LN000000018    1
LN000000019    1
LN000000020    1
Name: count, dtype: int64

 customer_id
customer_id
CU00411848    11
CU00289120    11
CU00565558    11
CU00460861    10
CU00775300    10
CU01265949    10
CU00603505    10
CU00466167    10
CU00036485    10
CU01282995     9
CU00782633     9
CU00338986     9
CU00963347     9
CU00074559     9
CU00524510     9
CU00212973     9
CU01143056     9
CU00812855     9
CU00104995     9
CU00243239     9
Name: count, dtype: int64

 branch_id
branch_id
BR7328-MH    52
BR6972-MH    49
BR6091-MH    48
BR1331-MH    48
BR6428-MH    48
BR9906-MH    47
BR9647-MH    47
BR5156-MH    46
BR8391-MH    46
BR8891-MH    46
BR6435-MH    46
BR1728-MH    46
BR8595-MH 

In [46]:
print("emi_to_income_ration")
print(merged["emi_to_income_ratio"].describe())
print("Values > 1:", (merged["emi_to_income_ratio"] > 1).sum())
print("Values < 0:", (merged["emi_to_income_ratio"] < 0).sum())

emi_to_income_ration
count    2.000000e+06
mean     2.127907e-01
std      3.274482e-01
min      5.000000e-04
25%      5.200000e-02
50%      1.129000e-01
75%      2.430000e-01
max      2.858450e+01
Name: emi_to_income_ratio, dtype: float64
Values > 1: 52798
Values < 0: 0


###
6. emit_to_income_ration = max=28.58 - No one pays 28 X income as EMI

In [56]:
# ── COMPLETE AUDIT: everything we haven't checked yet ─────────────────────

# 1. Cross-field logical consistency checks
print("="*60)
print("CROSS-FIELD CONSISTENCY CHECKS")
print("="*60)

# Funded amount should NEVER exceed loan amount
print("\n[1] funded_amnt > loan_amnt:")
print((merged["funded_amnt_inr"] > merged["loan_amnt_inr"]).sum())

# Disbursed should not exceed sanctioned
print("\n[2] disbursed_amount > sanctioned_amount:")
print((merged["disbursed_amount_inr"] > merged["sanctioned_amount_inr"]).sum())

# Total paid should not exceed total due
print("\n[3] total_emi_paid > total_emi_due (by large margin):")
excess = merged["total_emi_paid_inr"] - merged["total_emi_due_inr"]
print(excess.describe())
print(f"Rows where paid > due by >50k: {(excess > 50000).sum()}")

# Out principal should not exceed loan amount
print("\n[4] out_prncp_inr > loan_amnt_inr:")
print((merged["out_prncp_inr"] > merged["loan_amnt_inr"]).sum())

# Total payments should not exceed loan amount + interest by huge margin
print("\n[5] total_pymnt_inr > loan_amnt_inr * 3:")
print((merged["total_pymnt_inr"] > merged["loan_amnt_inr"] * 3).sum())

# EMI overdue cannot exceed total EMI due
print("\n[6] emi_overdue > total_emi_due:")
print((merged["emi_overdue_inr"] > merged["total_emi_due_inr"]).sum())

# ── 2. cibil_score vs cibil_score_band consistency ────────────────────────
print("\n" + "="*60)
print("CIBIL SCORE vs BAND CONSISTENCY")
print("="*60)
print(merged.groupby("cibil_score_band")["cibil_score"].agg(["min","max","mean","count"]))

# ── 3. Age vs employment length ───────────────────────────────────────────
print("\n" + "="*60)
print("AGE vs EMP_LENGTH CONSISTENCY")
print("="*60)
# Person aged 22 cannot have 15 years employment
impossible_emp = merged["emp_length_years"] > (merged["age"] - 18)
print(f"emp_length > (age - 18): {impossible_emp.sum()}")

# ── 4. Prepayment flag vs actual prepayment ───────────────────────────────
print("\n" + "="*60)
print("PREPAYMENT FLAG vs ADVANCE PAID")
print("="*60)
# Flag=1 but no advance paid
flag1_no_pay = ((merged["prepayment_flag"] == 1) & 
                (merged["emi_advance_paid_inr"] == 0)).sum()
print(f"prepayment_flag=1 but emi_advance_paid=0: {flag1_no_pay}")
# Flag=0 but advance paid exists  
flag0_has_pay = ((merged["prepayment_flag"] == 0) & 
                 (merged["emi_advance_paid_inr"] > 0)).sum()
print(f"prepayment_flag=0 but emi_advance_paid>0: {flag0_has_pay}")

# ── 5. OTS accepted without being offered ─────────────────────────────────
print("\n" + "="*60)
print("OTS LOGIC CHECK")
print("="*60)
ots_impossible = ((merged["ots_accepted_flag"] == 1) & 
                  (merged["ots_offered_flag"] == 0)).sum()
print(f"ots_accepted=1 but ots_offered=0: {ots_impossible}")

# ── 6. Waiver amount without waiver flag ──────────────────────────────────
print("\n" + "="*60)
print("WAIVER LOGIC CHECK")
print("="*60)
waiver_mismatch = ((merged["waiver_granted_flag"] == 0) & 
                   (merged["waiver_amount_inr"] > 0)).sum()
print(f"waiver_flag=0 but waiver_amount>0: {waiver_mismatch}")

# ── 7. NPA flag vs DPD bucket consistency ─────────────────────────────────
print("\n" + "="*60)
print("NPA FLAG vs DPD BUCKET")
print("="*60)
print(pd.crosstab(merged["npa_flag"], merged["dpd_bucket"]))

# ── 8. Loan secured flag vs collateral ───────────────────────────────────
print("\n" + "="*60)
print("LOAN SECURED vs HAS COLLATERAL")
print("="*60)
print(pd.crosstab(merged["loan_secured_flag"], merged["has_collateral"]))

# ── 9. has_credit_card=0 but cc fields have values ────────────────────────
print("\n" + "="*60)
print("NO CREDIT CARD BUT CC DATA EXISTS")
print("="*60)
no_card_has_data = ((merged["has_credit_card"] == 0) & 
                    (merged["total_cc_balance_inr"] > 0)).sum()
print(f"has_credit_card=0 but balance>0: {no_card_has_data}")

# ── 10. issue_year vs loan_term — would loan end in future? ───────────────
print("\n" + "="*60)
print("ISSUE YEAR + LOAN TERM SANITY")
print("="*60)
merged["loan_end_year"] = merged["issue_year"] + (merged["loan_term_months"] / 12)
print(merged["loan_end_year"].describe())
print(f"Loan ending after 2030: {(merged['loan_end_year'] > 2030).sum()}")

CROSS-FIELD CONSISTENCY CHECKS

[1] funded_amnt > loan_amnt:
0

[2] disbursed_amount > sanctioned_amount:
0

[3] total_emi_paid > total_emi_due (by large margin):
count    2.000000e+06
mean    -3.707377e+03
std      2.763848e+04
min     -1.998671e+06
25%     -2.839425e+03
50%     -1.925350e+02
75%      2.338890e+03
max      2.481048e+05
dtype: float64
Rows where paid > due by >50k: 519

[4] out_prncp_inr > loan_amnt_inr:
0

[5] total_pymnt_inr > loan_amnt_inr * 3:
0

[6] emi_overdue > total_emi_due:
0

CIBIL SCORE vs BAND CONSISTENCY
                  min  max        mean   count
cibil_score_band                              
Excellent         800  900  836.306430  159733
Fair              650  699  674.651287  461600
Good              700  749  723.200200  404650
Poor              550  649  608.128791  595222
Very Good         750  799  771.873859  254454
Very Poor         303  549  512.638240  124331

AGE vs EMP_LENGTH CONSISTENCY
emp_length > (age - 18): 180929

PREPAYMENT FLAG vs A

/tmp/ipykernel_5591/3708671371.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(merged.groupby("cibil_score_band")["cibil_score"].agg(["min","max","mean","count"]))


dpd_bucket  1-30 DPD  31-60 DPD  61-90 DPD  90+ DPD / NPA  Current
npa_flag                                                          
0              68417       8440        684              0  1922444
1                  0          0          0             15        0

LOAN SECURED vs HAS COLLATERAL
has_collateral           0       1
loan_secured_flag                 
0                  1258896       0
1                        0  741104

NO CREDIT CARD BUT CC DATA EXISTS
has_credit_card=0 but balance>0: 0

ISSUE YEAR + LOAN TERM SANITY
count    2.000000e+06
mean     2.021690e+03
std      4.341846e+00
min      2.011000e+03
25%      2.018000e+03
50%      2.022000e+03
75%      2.025000e+03
max      2.031000e+03
Name: loan_end_year, dtype: float64
Loan ending after 2030: 8141
